# Linear Projection (GEMM)
*Matrix multiplication — the most compute-intensive operation in a transformer.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_smem_tiled.py` → `sm89_v2_tensor_core_mma.py`


## What Is a Linear Projection?

**In plain English:** A linear layer multiplies an input matrix by a weight matrix. This is how the model transforms representations: attention projections (Q, K, V, output), feedforward up/down projections — every "thinking" step is a matrix multiply.

**Where it appears:** Every attention sublayer (4× matmuls for Q, K, V, output projection) + every FFN sublayer (2-3× matmuls). ~70% of a transformer's total FLOPs are matrix multiplies.


In [ ]:
import math, random
random.seed(42)
print('ready')

## Worked Example: $2 \times 3$ Output from $2 \times 2$ Input

$$X = \begin{bmatrix} 1.0 & 2.0 \\ 3.0 & 4.0 \end{bmatrix}, \quad W = \begin{bmatrix} 0.5 & 0.5 \\ 0.3 & -0.2 \\ -0.1 & 0.8 \end{bmatrix}$$

We compute $Y = X W^T$ (PyTorch `F.linear` convention: W rows = output directions).

$$Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}$$

| $Y[m,n]$ | Dot product | Result |
|-----------|-------------|--------|
| $Y[0,0]$ | $1.0×0.5 + 2.0×0.5$ | **1.5** |
| $Y[0,1]$ | $1.0×0.3 + 2.0×(-0.2)$ | **-0.1** |
| $Y[0,2]$ | $1.0×(-0.1) + 2.0×0.8$ | **1.5** |
| $Y[1,0]$ | $3.0×0.5 + 4.0×0.5$ | **3.5** |
| $Y[1,1]$ | $3.0×0.3 + 4.0×(-0.2)$ | **0.1** |
| $Y[1,2]$ | $3.0×(-0.1) + 4.0×0.8$ | **2.9** |


In [ ]:
X = [[1.0, 2.0], [3.0, 4.0]]
W = [[0.5, 0.5], [0.3, -0.2], [-0.1, 0.8]]
M, K, N = 2, 2, 3

Y = [[sum(X[m][k] * W[n][k] for k in range(K)) for n in range(N)] for m in range(M)]

print("Y = X @ W.T:")
for m, row in enumerate(Y):
    print(f"  row {m}: {[round(v,4) for v in row]}")

# Spot-check
assert abs(Y[0][0] - 1.5) < 1e-9 and abs(Y[1][2] - 2.9) < 1e-9
print("\n✓ Spot-checks pass")


## The Formula

$$\boxed{Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}}$$

Equivalently: $Y = X W^T$ where $X \in \mathbb{R}^{M \times K}$, $W \in \mathbb{R}^{N \times K}$, $Y \in \mathbb{R}^{M \times N}$.

| Symbol | Meaning | In our example |
|--------|---------|----------------|
| $M$ | Batch size (number of input vectors) | 2 |
| $N$ | Output dimension (number of neurons) | 3 |
| $K$ | Input dimension | 2 |
| $X_{m,k}$ | Element $k$ of input vector $m$ | $X[0,0]=1.0$ |
| $W_{n,k}$ | Weight for output $n$ from input $k$ | $W[0,0]=0.5$ |

### 🗣️ Say It Out Loud
> *"Y at row m, column n equals the sum over k of X at m-comma-k times W at n-comma-k. This is the dot product of input row m with weight row n."*

**Tutor's gloss:** "Think of each row of W as a 'detector'. $W[0,:]$ detects the pattern $[0.5, 0.5]$ — it responds to the average of the two input features. The dot product with $X[0,:]=[1,2]$ gives $0.5×1 + 0.5×2 = 1.5$ — the detector fired."


## Arithmetic Intensity and the Cache Problem

**Think of a GPU as a factory.**

The GPU has two parts: a **warehouse** (HBM — holds all weights and activations) and a **factory floor** (compute units: tensor cores). Every computation follows the same cycle: fetch raw materials from the warehouse, process them, ship results back. The factory floor is extremely fast; the warehouse delivery pipeline is the bottleneck.

**Arithmetic intensity** measures how much factory work gets done per unit of warehouse delivery:

$$\text{Arithmetic intensity} = \frac{\text{FLOPs done}}{\text{bytes moved from HBM}}$$

A factory that burns 1 ton of material to produce 1 gram of product is *supply-limited* — adding faster machines won't help if the warehouse trucks can't keep up. The RTX 4080 sits at a "ridge point" of roughly 151 FLOP/byte. Below the ridge, HBM bandwidth is the bottleneck; above it, tensor cores are.

**For matrix multiplication `Y = X @ W.T`** with `X: [M, K]`, `W: [N, K]`, `Y: [M, N]`:
- FLOPs: `2·M·N·K` (one multiply-add per element of each inner product)
- Bytes moved (minimum): `2·(M·K + N·K + M·N)` bytes to read X, read W, write Y

For large square matrices with side n:
$$I \approx \frac{2n^3}{6n^2 \times 2} = \frac{n}{6} \text{ FLOP/byte}$$

| Matrix size | Intensity | Verdict on RTX 4080 (ridge ≈ 151 F/B) |
|---|---|---|
| 64 × 64 | ~10 FLOP/byte | Memory-bound — tensor cores mostly idle |
| 1024 × 1024 | ~170 FLOP/byte | Just above the ridge — compute-bound |
| 4096 × 4096 | ~680 FLOP/byte | Strongly compute-bound |
| M=1 (decode) | ~0.5 FLOP/byte | 300× below ridge — weight reads dominate everything |

At decode (M=1, generating one token), intensity is ~0.5 FLOP/byte regardless of the weight matrix size. No tiling trick or tensor-core optimization helps here — this is a supply problem, not a factory problem. The only effective lever is delivering *fewer bytes* (quantization: FP8 halves the weight sizes → 2× faster loads → ~2× faster decode).

---

### The cache problem in `v0_naive`

In the naive kernel, each output element `Y[m, n]` is computed by its own thread. That thread loads its row of X from HBM and then, for each output column n, fetches row `W[n, :]` from HBM. With N=4096 output columns, every thread producing output in row m fetches all 4096 weight rows — and so does every other thread. Row 0 of W is loaded by every thread in the grid computing column 0. Row 1 is loaded by every thread computing column 1. At M=2048:

**Row 0 of W is fetched 2048 separate times from HBM.** Every fetch is identical.

This is pure waste: HBM bandwidth spent on redundant reads of the same data.

**The fix — tiling:** Assign a block of `B_M × B_N` threads to cooperate. They collectively load a `B_M × B_K` tile of X and a `B_N × B_K` tile of W into shared memory *once*, then all `B_M × B_N` threads compute from that on-chip copy. One HBM fetch, used `B_M` times. At `B_M = B_N = 128`: each weight byte produces 128 outputs instead of 1. This is the jump from `v0_naive` (<0.1 TFLOPS) to `v1_smem_tiled` (1–5 TFLOPS).

## Optimization Ladder

| Version | Compute path | Reuse | Approx. TFLOPS |
|---------|-------------|-------|----------------|
| `v0_naive` | CUDA scalar FMA | None | < 0.1 |
| `sm89_v1_smem_tiled` | CUDA scalar FMA | $B_M$ and $B_N$ | 1–5 |
| `sm89_v2_tensor_core_mma` | m16n8k16 TC instruction | $B_M \times B_N$ | 50–165 |

**Tensor Core instruction (SM89):**
`D[16,8] += A[16,16] × B[16,8]` in fp16 — 4096 FLOPs per warp per cycle.
In CuTeDSL: `cute.gemm(tiled_mma, rA, rB, rC)` emits this PTX.


## RTX 4080: Two Very Different GEMM Regimes

The same matrix multiply formula hits the RTX 4080 in two completely different ways depending on batch size.

**Decode mode (M=1, one token at a time):**
One input row × weight matrix = a GEMV (vector-matrix multiply).
- Data moved: read W (N×K values) + read x (K values) + write y (N values)
- FLOPs: 2·N·K
- For q_proj (N=K=4096): 2·4096·4096 FLOPs / (2·4096·4096·2 bytes) ≈ **0.5 FLOP/byte**
- Far below the ridge (151 FLOP/byte) → **memory-bound**, tensor cores sit idle
- Speedup path: FP8 cuts weight bytes in half → 2× faster reads

**Prefill mode (M=seq_len, processing the prompt):**
Full matrix multiply. As M grows, reuse of W improves.
- At M=512: intensity ≈ 512·4096·4096 / (512·4096·2 + 4096·4096·2 + 512·4096·2) bytes ≈ **85 FLOP/byte** (approaching ridge)
- At M=2048: intensity ≈ **340 FLOP/byte** → firmly compute-bound
- Speedup path: maximize tensor core utilization (tiling, async copies, persistent kernels)

| Regime | M | Intensity | Bound | Key speedup |
|--------|---|-----------|-------|-------------|
| Decode | 1 | 0.5 F/B | Memory | FP8 weights (2×) |
| Prefill small | 32 | 10 F/B | Memory | Better tiling |
| Prefill large | 2048 | 340 F/B | Compute | Tensor cores (57.5 TFLOPS) |

## CuTeDSL: Tensor Core MMA Walkthrough

The jump from `v1_smem_tiled` to `v2_tensor_core_mma` is the biggest speedup in the whole ladder.
Here is what that looks like:

```python
# Define the MMA "atom" — the smallest tensor core operation available on SM89
# SM89_16x8x16: processes a 16×16 fragment of A and 8×16 fragment of B
# Result: a 16×8 fragment of C, accumulated in FP32
tiled_mma = cute.make_tiled_mma(
    cute.SM89_16x8x16_F32BF16BF16F32_TN()
    # Read this name as: 16 output rows, 8 output cols, 16 inner depth
    # F32   = accumulator type (FP32 for precision)
    # BF16  = A matrix type
    # BF16  = B matrix type
    # F32   = output type
    # T = A is transposed in memory, N = B is not
)

@cute.kernel
def gemm_tc_kernel(mX, mW, mOut):
    # Each CTA (block) owns a [Bm × Bn] tile of the output
    rC = cute.partition_fragment_C(tiled_mma, ...)  # output accumulator in registers

    for k_tile in range(K // Bk):
        # Step 1: cooperatively load a [Bm × Bk] tile of X into shared memory
        # All 128+ threads in the CTA participate — each loads a few elements
        cute.copy(mX[row_tile, k_tile], sX)   # HBM → SMEM
        cute.copy(mW[col_tile, k_tile], sW)   # HBM → SMEM
        cute.sync_threads()  # everyone must finish loading before computing

        # Step 2: each thread loads its register fragment from shared memory
        cute.copy(sX, rA)
        cute.copy(sW, rB)

        # Step 3: fire the tensor core instruction
        # This single line compiles to PTX `mma.sync.aligned.m16n8k16`
        # One GPU clock cycle, 32 threads acting in lockstep → 4,096 FLOPs
        cute.gemm(tiled_mma, rA, rB, rC)   # rC += rA × rB

        cute.sync_threads()

    # Write the result from registers back to HBM
    cute.copy(rC, mOut[row_tile, col_tile])
```

**Why this is so much faster than scalar FMA:**

In `v1_smem_tiled`, each thread does its own multiply-adds one at a time.
In `v2_tensor_core_mma`, a group of 32 threads (a "warp") all fire ONE instruction together that does a 16×8×16 matrix multiply in hardware.

The SM89 tensor core is custom silicon designed to do exactly this — multiply two small matrices in one shot. No branching, no loop overhead, just pure math.

The RTX 4080 has 76 SMs × ~4 tensor core units each = hundreds of these firing simultaneously.
At peak: 57.5 TFLOPS BF16. With FP8: 115 TOPS (the operands are half the size so twice as many fit per cycle).

## Tile Sizing for Qwen3-8B on RTX 4080

Choosing tile dimensions is the most important decision in a GEMM kernel. Too small and you don't get enough data reuse. Too large and you can't fit in shared memory or run enough CTAs in parallel.

**RTX 4080 constraints:**
- 128 KB shared memory per SM (configurable — can dedicate up to ~100 KB to SMEM)
- 255 registers per thread, up to 1024 threads per SM
- 76 SMs total

**Standard tile for BF16 GEMM on SM89:** BM=128, BN=128, BK=32

```
SMEM per CTA (single buffer):
  A tile [BM=128, BK=32] in BF16  = 128×32×2 = 8 KB
  B tile [BN=128, BK=32] in BF16  = 128×32×2 = 8 KB
  Total: 16 KB

With double buffering (overlap load with compute):
  A×2 + B×2 = 32 KB
  128 KB / 32 KB = 4 CTAs per SM  ← good occupancy
```

**Worked example: q_proj at prefill (M=2048, N=4096, K=4096)**
```
Grid: (M/BM, N/BN) = (2048/128, 4096/128) = (16, 32) = 512 CTAs
K-tiles per CTA: K/BK = 4096/32 = 128 tiles
Each CTA: 128×128 output elements, 128 K-tiles
Total MMA ops per CTA: (128/16) × (128/8) × (128/16) = 8×16×8 = 1024 m16n8k16 ops
76 SMs × 4 CTAs = 304 active at once  →  512/304 ≈ 2 waves → GPU nearly fully utilized
```

**Worked example: k_proj at decode (M=1, N=1024, K=4096)**

This is a GEMV. The standard tiled GEMM doesn't help here — a 128×128 output tile with M=1 leaves 127/128 of the tile empty.

**Suggestion: split-K for decode GEMV**
```
Split K=4096 into 4 chunks of 1024. Each chunk is one "split":
  Split 0: computes partial y[0:1024] using W[0:1024, 0:4096/4]
  Split 1: computes partial y[0:1024] using W[0:1024, 1024:2048]
  ...
4 CTAs run in parallel, each computing a partial dot product over K/4 elements.
Then one final reduction: add the 4 partial results.

Result: 4× more parallelism → 4× better SM utilization during decode.
```

| Shape | M | N | K | Strategy | CTAs | Notes |
|-------|---|---|---|----------|------|-------|
| q_proj prefill | 2048 | 4096 | 4096 | Tiled (128×128) | 512 | GPU saturated |
| q_proj decode | 1 | 4096 | 4096 | Split-K (4 splits) | 4×32 = 128 | Acceptable |
| k_proj decode | 1 | 1024 | 4096 | Split-K (4 splits) | 4×8 = 32 | Underutilized |
| down_proj decode | 1 | 4096 | 12288 | Split-K (12 splits) | 12×32 = 384 | Good |

## Double Buffering: Overlapping Memory and Compute

### The restaurant kitchen analogy

Imagine a chef (tensor cores) and a prep assistant (memory system). The kitchen has two prep stations, A and B.

**Without double buffering:** The chef must wait for the assistant to finish loading station A, then cooks, then waits again while the assistant clears A and loads station B. The chef idles every time the assistant works.

```
Timeline:
  [──prep A──] [cook A] [──prep B──] [cook B] [──prep C──] [cook C]
               ↑ idle              ↑ idle              ↑ idle
```

**With double buffering:** While the chef cooks from station A, the assistant loads station B in parallel. When A is done, the chef immediately moves to B — no waiting — while the assistant restocks A.

```
Timeline:
  [──prep A──] [cook A] [cook B] [cook C]
               [──prep B──] [──prep C──]
               ↑ no idle   ↑ no idle
```

The RTX 4080 makes this possible with `cp.async` — an instruction that copies data from global memory to shared memory *in the background* while the SM keeps computing. Without `cp.async`, a load blocks: the SM sits idle waiting for HBM.

---

### The SMEM layout: ping-pong between two slots

```
SMEM for double-buffered GEMM (BM=128, BN=128, BK=32, BF16):
  sA_0 [128 × 32] in BF16  = 8 KB  ← station A for A tiles
  sA_1 [128 × 32] in BF16  = 8 KB  ← station B for A tiles
  sB_0 [128 × 32] in BF16  = 8 KB  ← station A for B tiles
  sB_1 [128 × 32] in BF16  = 8 KB  ← station B for B tiles
  Total: 32 KB
```

Even k-tiles use slot 0; odd k-tiles use slot 1. The slot being consumed by tensor cores is simultaneously being refilled from HBM by `cp.async`.

---

**CuTeDSL pipeline code:**

```python
# Two-stage pipeline: ping-pong between slots 0 and 1
pipeline = cute.make_pipeline(stages=2, smem_a=sA_double, smem_b=sB_double)

# Prologue: fire the very first tile load BEFORE entering the loop
# (station A is prepped before the chef even walks in)
cute.pipeline_producer_acquire(pipeline, 0)
cute.copy(gmem_tile_A[0], pipeline.smem_a(0))   # cp.async: fires in background, returns immediately
cute.copy(gmem_tile_B[0], pipeline.smem_b(0))   # cp.async: fires in background, returns immediately
cute.pipeline_producer_commit(pipeline, 0)

for k_tile in range(num_k_tiles):
    # "Chef arrives at the station" — stall until this tile has fully landed in SMEM
    cute.pipeline_consumer_wait(pipeline, k_tile)

    # Load tile from SMEM into registers using ldmatrix.
    # ldmatrix.sync.aligned.m8n8.x4.shared.b16: one PTX instruction that reads
    # 128 bytes from SMEM and distributes them across all 32 threads in the warp,
    # in the exact byte layout the tensor core expects — no manual index math.
    cute.copy(smem_to_rmem_copy, pipeline.smem_a(k_tile), rA)
    cute.copy(smem_to_rmem_copy, pipeline.smem_b(k_tile), rB)

    # "Prep assistant starts loading station B" — fire background copy of NEXT tile
    if k_tile + 1 < num_k_tiles:
        cute.pipeline_producer_acquire(pipeline, k_tile + 1)
        cute.copy(gmem_tile_A[k_tile+1], pipeline.smem_a(k_tile+1))
        cute.copy(gmem_tile_B[k_tile+1], pipeline.smem_b(k_tile+1))
        cute.pipeline_producer_commit(pipeline, k_tile + 1)

    # "Chef cooks from station A" — tensor cores fire while the next tile loads
    cute.gemm(tiled_mma, rA, rB, rC)   # rC += rA × rB

    cute.pipeline_consumer_release(pipeline, k_tile)

cute.copy(rC, mOut[row_tile, col_tile])
```

---

**ldmatrix — why it exists:**

Before `cute.gemm` fires, each of the 32 threads in a warp needs its portion of the A and B tile fragments in registers, laid out in the precise byte order the tensor core hardware requires. Getting this right by hand means careful index arithmetic that varies by tile size and data type — error-prone and unreadable.

The PTX instruction `ldmatrix.sync.aligned.m8n8.x4.shared.b16` handles this automatically: it reads an aligned BF16 matrix fragment from SMEM and broadcasts the bytes across the warp in one shot, already in tensor-core order. In CuTeDSL, `smem_to_rmem_copy` above resolves to this instruction — you express what you want (SMEM tile → register fragment), CuTeDSL emits the correct PTX.

---

**RTX 4080 numbers with BM=128, BN=128, BK=32, double buffered:**

```
SMEM per CTA:      2 × (8 KB + 8 KB) = 32 KB
Registers/thread:  ~128 (rA=16, rB=8, rC=32, plus overhead)
CTAs per SM:       128 KB ÷ 32 KB = 4  ← target occupancy for SM89 BF16 GEMMs
Active CTAs:       76 SMs × 4 = 304 simultaneously
```

A prefill GEMM for q_proj (M=2048, N=K=4096) launches 512 output tiles and completes in ~2 waves — near-perfect GPU utilization.

## Where GEMM Lives: The Qwen3-8B Projection Weights

Matrix multiplications account for ~95% of the FLOPs in a Qwen3-8B forward pass.
Here is every GEMM in one decode step, with its exact shape and memory footprint.

### The 7 weight GEMMs per layer (× 36 layers)

```
┌──────────────────────────────────────────────────────────────────────────┐
│  x: [1, 4096]  ← token hidden state (one row for decode)               │
│                                                                          │
│  ① Q projection  W[4096, 4096]   x @ W.T  → q [1, 4096]  32 MB BF16   │
│  ② K projection  W[1024, 4096]   x @ W.T  → k [1, 1024]   8 MB BF16   │
│  ③ V projection  W[1024, 4096]   x @ W.T  → v [1, 1024]   8 MB BF16   │
│  ...RoPE, QK-norm, attention...                                          │
│  ④ O projection  W[4096, 4096]  attn @ W.T → [1, 4096]   32 MB BF16   │
│                                                                          │
│  ⑤ gate_proj    W[12288, 4096]  x @ W.T  → g [1, 12288] 100 MB BF16   │
│  ⑥ up_proj      W[12288, 4096]  x @ W.T  → u [1, 12288] 100 MB BF16   │
│     SwiGLU(g, u) → hidden [1, 12288]                                    │
│  ⑦ down_proj    W[4096, 12288]  h @ W.T  → [1, 4096]   100 MB BF16    │
└──────────────────────────────────────────────────────────────────────────┘
```

Plus 2 non-layer operations:
- Embedding lookup: token_id → `[1, 4096]` (a gather, not a matmul)
- LM head: `[1, 4096] × [151936, 4096].T → [1, 151936]` (1.2 GB weights — the largest single weight in the model)

### Weight reads per decode step (batch=1)

```
Per layer:  32 + 8 + 8 + 32 + 100 + 100 + 100 = 380 MB BF16
× 36 layers: 13.7 GB
+ LM head:    1.2 GB
─────────────────
Total:       ~15 GB per token generated
```

At 380 GB/s: ~39 ms per token → ~25 tokens/second at BF16 decode.

### What "M=1 means the tensor core sits idle"

At decode, M=1 means the tile shape is `[1, N, K]`. The standard GEMM tile is `[BM, BN, BK] = [128, 128, 32]`. A `[1, 128, 32]` tile is 127/128 wasted. Tensor cores fire but most lanes produce dummy output.

Arithmetic intensity at decode: `2×1×N×K / (1×K×2 + N×K×2 + 1×N×2) bytes ≈ 0.5 FLOP/byte` — 300× below the ridge.

### Why prefill looks completely different

At prefill M=2048: the grid is `(16, 32) = 512 CTAs` — every SM is busy.
Intensity rises to ~340 FLOP/byte. Tensor cores dominate. Memory latency is hidden by compute.
This is why prefill is 10–100× more efficient per token than decode on the RTX 4080.

## GEMM Implementations in the Wild

### cuBLAS / cuBLASLt

The default for any framework calling `torch.nn.Linear`.
- `cublasGemmEx` for general mixed-precision GEMMs
- `cublasLtMatmul` adds fused epilogues: bias add, ReLU, GELU folded into the GEMM kernel
- SM89: FP8 support via `CUDA_R_8F_E4M3` / `CUDA_R_8F_E5M2` data types
- Black box: no PTX visibility, tile sizes chosen by heuristic

### CUTLASS 3.x (and CuTe)

The template library this project uses.
- Exposes the full tiling hierarchy: `TiledMma`, `TiledCopy`, `PipelinedGemm`
- SM89 `SM89_16x8x16_F32BF16BF16F32_TN` is the atom used in `sm89_v2_tensor_core_mma.py`
- Persistent kernel style available via `cutlass::gemm::device::GemmUniversal`

### FlashInfer

- `flashinfer.gemm.bmm_fp8` — batched GEMM with FP8 inputs and per-tensor or per-block scales
- `flashinfer.activation.silu_and_mul` — fused SwiGLU after the gate+up GEMMs
- Separates the GEMM from the epilogue so fusions are composable

### vLLM

- `vllm/csrc/ops/gemm_kernels.cu` and the Marlin kernel for W4A16 quantized GEMMs
- `vllm/csrc/ops/fp8_gemm.cu` wraps cuBLASLt with FP8 block scales
- The "paged linear" trick: W is stored contiguously but activations come from paged memory

### TensorRT-LLM

- Compiles entire attention+norm+GEMM chains into a single TRT plugin
- The `FusedMHA` plugin fuses Q/K/V projection → attention → O projection into one kernel
- Also supports the GPTQ quantization format (W4A16) via Marlin-style interleaved layouts

### Pattern across all implementations

Every production GEMM framework does the same four things:
1. **Double buffering** — overlap HBM loads with tensor core compute via `cp.async`
2. **Swizzled SMEM layout** — avoid bank conflicts when threads load to registers
3. **Fused epilogues** — bias / norm / activation in the same kernel, not a separate pass
4. **Split-K for decode** — parallelize over the K dimension when M=1 to improve SM utilization

## One-Sentence Takeaway

In Qwen3-8B inference, matrix multiplication accounts for ~95% of FLOPs and ~90% of weight bytes; at decode the GPU is memory-bound (reads ~15 GB of weights per token) making weight quantization the single highest-leverage optimization, while at prefill the GPU becomes compute-bound at M ≥ 512 making tensor-core utilization and tile configuration the critical knobs — and the architectural gap between these two regimes (GEMV vs GEMM, 0.5 FLOP/byte vs 340 FLOP/byte) is why production serving systems use different kernels for decode and prefill and why Split-K matters so much for single-token generation.

## Community Optimization Resources for Linear Projection (GEMM) on SM89

### Speedup reference (community benchmarks on SM89-adjacent hardware)

| Optimization | Hardware | Result | Implementation | Notes |
|---|---|---|---|---|
| BF16 GEMM vs cuBLAS | RTX 3070 Laptop (SM86) | **−6.77%** cuBLAS gap after 311 rounds | Triton | Auto-tuned Triton reaches 93.2% of cuBLAS |
| Triton GEMM (matmul_optimizer) | RTX 4090 (SM89) | **80–95%** of cuBLAS | Triton | Best config: BLOCK_M=128, BLOCK_N=128, num_stages=4 |
| Split-K for decode GEMV | RTX 3090 (SM86) | **~3× SM utilization** vs naive GEMV | Triton | M=1, K=4096 split into 4 chunks |
| CUTLASS GemmUniversal BF16 | RTX 4090 (SM89) | **>95%** of cuBLAS | CUTLASS | threadblock swizzle adjustable on SM89 |
| TritonForge auto-tuner | RTX 4090 (SM89) | **91–97%** of cuBLAS | Triton | Grid search over BLOCK_M/N/K + num_stages |

**Key finding:** On SM89, well-tuned Triton GEMMs reach 80–95% of cuBLAS. The gap widens for small M (decode GEMV) and narrows for large square matrices (prefill).

---

### SM89 MMA atom: the tensor core instruction

The fundamental compute unit for BF16 GEMM on SM89 Ada Lovelace:

```python
# CuTeDSL atom name decodes as:
# SM89 _ 16×8×16 _ F32BF16BF16F32 _ TN
#  ↑       ↑           ↑              ↑
#  arch  tile      precision types   layout
#                 (accum, A, B, out)  (T=transposed, N=non-transposed)

tiled_mma = cute.make_tiled_mma(
    cute.SM89_16x8x16_F32BF16BF16F32_TN()
    # PTX instruction: mma.sync.aligned.m16n8k16.row.col.f32.bf16.bf16.f32
    # 4,096 FLOPs per warp per clock cycle
    # 32 threads participate simultaneously, each loading 8 A elements + 4 B elements
)
```

For FP8 (double the throughput):
```python
cute.SM89_16x8x32_F32E4M3E4M3F32_TN()  # k=32, 8,192 FLOPs/warp/cycle
```

---

### SMEM budget formula for BF16 GEMM (double-buffered)

```
2 × (BM × BK + BN × BK) × 2 bytes ≤ 99 KB   (RTX 4080: 99 KB usable per block)

At BM=128, BN=128, BK=64:
  2 × (128×64 + 128×64) × 2 = 2 × 16384 × 2 = 65,536 bytes = 64 KB  ✅

At BM=128, BN=128, BK=128:
  2 × (128×128 + 128×128) × 2 = 2 × 32768 × 2 = 131,072 bytes = 128 KB  ❌ exceeds limit
```

**BK=64 is the maximum double-buffered tile depth on SM89.** Triton configs that specify `BLOCK_K=128` with double buffering will silently fall back to single buffering.

---

### TritonForge → CuTeDSL parameter mapping

| TritonForge / Triton parameter | CuTeDSL equivalent | Notes |
|---|---|---|
| `BLOCK_M` | `BM` (tile rows) | Must be multiple of 16 (MMA atom height) |
| `BLOCK_N` | `BN` (tile cols) | Must be multiple of 8 (MMA atom width) |
| `BLOCK_K` | `BK` (tile depth) | Max 64 with double buffering |
| `num_stages=2` | `cute.make_pipeline(stages=2)` | Explicit `cp.async` commit/wait groups |
| `num_stages=4` | `cute.make_pipeline(stages=4)` | 4× SMEM — only works with small BK |
| `num_warps=4` | warp layout `(2,2)` | 4 warps per block, 128 threads total |
| `num_warps=8` | warp layout `(2,4)` or `(4,2)` | 256 threads — check register pressure |

---

### Projection shapes in Qwen3-8B: what to tune for

| Projection | Shape (decode M=1) | Shape (prefill M=2048) | Key challenge |
|---|---|---|---|
| q_proj | [1, 4096] × [4096, 4096] | [2048, 4096] × [4096, 4096] | Decode: GEMV, use Split-K |
| k_proj | [1, 4096] × [4096, 1024] | [2048, 4096] × [4096, 1024] | Smaller N, less parallelism |
| v_proj | same as k_proj | same as k_proj | Same as k_proj |
| o_proj | [1, 4096] × [4096, 4096] | [2048, 4096] × [4096, 4096] | Same as q_proj |
| gate_proj | [1, 4096] × [4096, 12288] | [2048, 4096] × [4096, 12288] | Large N — good parallelism |
| up_proj | same as gate_proj | same as gate_proj | Can fuse with gate_proj |
| down_proj | [1, 12288] × [12288, 4096] | [2048, 12288] × [12288, 4096] | Large K — Split-K at decode |

**Decode bottleneck:** 7 GEMMs × 380 MB avg weights × 36 layers = ~13.7 GB read per token. At 380 GB/s: ~36 ms/token baseline. FP8 cuts this to ~18 ms → the single highest-leverage optimization.

**References:**
- `openai/triton` — `python/tutorials/03-matrix-multiplication.py` (tiled GEMM tutorial)
- `CUTLASS 3.x` — `examples/48_hopper_warp_specialized_gemm` (SM89 equivalent: `examples/47_ampere_gemm_universal`)
- `TritonForge` (community) — grid-search auto-tuner for Triton GEMM configs on RTX 4090